## tl;dr

总回复 token 5939 → 5823；长尾和微批次不同，维持 REVIEW_WORKLOAD_DIFFERENCE。

## Context & Methods

两组各 64 条、8 步。以步骤＋唯一解码输入连接。

### Key Assumptions

日志 response_mask 均值×8 重建每步总 token；解码文本不能替代原始 token IDs。

In [1]:
import sys, json
from pathlib import Path
sys.path.insert(0, '/root/autodl-tmp/Vision-OPD-main')
from scripts.review_memory_ab_workload import review
comparison_path = Path('/root/autodl-tmp/Vision-OPD-main/artifacts/runs/E-D11-6K-GATE-001/memory_optimization/ab/deferred_v2/comparison_after_run.json')

## Data

校验原审计输入哈希、8 步日志指标、64 条输入配对及完整覆盖。

In [2]:
r = review(comparison_path)
assert all(r["checks"].values())
print(json.dumps(r["checks"], indent=2))

{
  "original_audit_inputs_unchanged": true,
  "independent_log_metric_match": true,
  "unique_sample_ids_64": true,
  "selected_ids_order_match": true,
  "per_step_decoded_input_set_match": true,
  "joined_ground_truth_match": true,
  "forward_sample_coverage_64": true
}


## Results

逐步长度与总体复算：

In [3]:
for row in r["step_rows"]:
    print(row)
for variant in ("baseline", "deferred"):
    total = sum(x[variant+"_response_mean"] * x["samples"] for x in r["step_rows"])
    assert total == r["total_response_tokens"][variant]
    print(variant, "tokens", total, "mean", total/64)
print("forward shapes:", r["shape_summary"])
print("identical decoded outputs:", r["joined_output_text_equal"])

{'step': 1, 'samples': 8, 'baseline_prompt_max': 7880.0, 'deferred_prompt_max': 7880.0, 'baseline_response_mean': 115.0, 'deferred_response_mean': 122.0, 'baseline_response_max': 255.0, 'deferred_response_max': 284.0, 'baseline_tokens': 920.0, 'deferred_tokens': 976.0, 'identical_output_texts': 2, 'baseline_student_microbatches': 6, 'deferred_student_microbatches': 6}
{'step': 2, 'samples': 8, 'baseline_prompt_max': 7318.0, 'deferred_prompt_max': 7318.0, 'baseline_response_mean': 69.875, 'deferred_response_mean': 84.625, 'baseline_response_max': 143.0, 'deferred_response_max': 159.0, 'baseline_tokens': 559.0, 'deferred_tokens': 677.0, 'identical_output_texts': 2, 'baseline_student_microbatches': 6, 'deferred_student_microbatches': 6}
{'step': 3, 'samples': 8, 'baseline_prompt_max': 7032.0, 'deferred_prompt_max': 7032.0, 'baseline_response_mean': 91.25, 'deferred_response_mean': 81.875, 'baseline_response_max': 229.0, 'deferred_response_max': 263.0, 'baseline_tokens': 730.0, 'deferred_t

## Takeaways

同一批输入，但生成和计算负载未受控。不能按 token 比例线性归一化显存，也不能解除长回复和 warmup 后验证。逐样本 token 分布缺失是明确限制，不是已完成验证。